# IMDb Plot → Genre Classifier (DistilBERT)

Fine-tunes `distilbert-base-uncased` on plot summaries from [`kolodkin/imdb-wikipedia-enriched`](https://huggingface.co/datasets/kolodkin/imdb-wikipedia-enriched) to predict genres as a multi-label classification problem.

- **Labels**: top 15 IMDb genres + `Other`
- **Split**: 80 / 10 / 10, multi-label stratified, deduped on plot hash, seeded
- **Eval**: macro-F1 / micro-F1 / per-genre P/R/F1 with bootstrap CI
- **Output**: pushed to [`kolodkin/imdb-genre-distilbert`](https://huggingface.co/kolodkin/imdb-genre-distilbert)
- **Runtime**: ~15 min on a Colab free T4


## Install

In [ ]:
!pip install -q transformers datasets huggingface_hub accelerate iterative-stratification scikit-learn tabulate


## Imports + config

In [ ]:
import hashlib
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import f1_score, precision_recall_fscore_support
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
MODEL_NAME = "distilbert-base-uncased"
DATASET_NAME = "kolodkin/imdb-wikipedia-enriched"
HF_REPO = "kolodkin/imdb-genre-distilbert"
TOP_K_GENRES = 15
MAX_LENGTH = 256
NUM_EPOCHS = 3
BATCH_SIZE = 32
LR = 5e-5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))


## Authenticate to Hugging Face

A write token is required to push the trained model. Create one at https://huggingface.co/settings/tokens.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()


## Load dataset

In [ ]:
ds = load_dataset(DATASET_NAME, split="train")
print(f"Rows: {len(ds):,}")
print(f"Columns: {ds.column_names}")
ds[0]


## Clean + dedupe

In [ ]:
def normalize_plot(s):
    return " ".join((s or "").lower().split())

def plot_hash(s):
    return hashlib.md5(normalize_plot(s).encode()).hexdigest()

df = ds.to_pandas()
before = len(df)

df = df[df["plot"].fillna("").str.len() >= 50]
df = df[df["genres"].apply(lambda g: isinstance(g, (list, np.ndarray)) and len(g) > 0)]
df["_hash"] = df["plot"].map(plot_hash)
df = df.drop_duplicates(subset="_hash").drop(columns="_hash").reset_index(drop=True)

print(f"Kept {len(df):,} / {before:,} rows after cleaning")


## Label space — top 15 genres + Other

In [ ]:
genre_counts = Counter(g for genres in df["genres"] for g in genres)
top = [g for g, _ in genre_counts.most_common(TOP_K_GENRES)]
print("Top genres:", top)

LABELS = top + ["Other"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
NUM_LABELS = len(LABELS)

def encode_labels(genres):
    vec = np.zeros(NUM_LABELS, dtype=np.float32)
    has_top = False
    for g in genres:
        if g in label2id:
            vec[label2id[g]] = 1.0
            has_top = True
    if not has_top:
        vec[label2id["Other"]] = 1.0
    return vec

df["labels"] = df["genres"].apply(encode_labels)

label_matrix = np.stack(df["labels"].values)
print("\nLabel frequencies:")
for label, count in zip(LABELS, label_matrix.sum(axis=0).astype(int)):
    print(f"  {label:<15} {count:>6,}")


## Stratified 80 / 10 / 10 split

In [ ]:
X = np.arange(len(df))
y = np.stack(df["labels"].values)

# 80 / 20 first
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, holdout_idx = next(msss.split(X, y))

# split 20 → 10 / 10
msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED)
val_rel, test_rel = next(msss2.split(holdout_idx, y[holdout_idx]))
val_idx = holdout_idx[val_rel]
test_idx = holdout_idx[test_rel]

print(f"train: {len(train_idx):,}  |  val: {len(val_idx):,}  |  test: {len(test_idx):,}")
assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0


## Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf(idx):
    sub = df.iloc[idx][["plot", "labels"]].reset_index(drop=True)
    return Dataset.from_pandas(sub)

train_ds = to_hf(train_idx)
val_ds = to_hf(val_idx)
test_ds = to_hf(test_idx)

def tokenize(batch):
    return tokenizer(batch["plot"], truncation=True, max_length=MAX_LENGTH)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=["plot"])
val_ds = val_ds.map(tokenize, batched=True, remove_columns=["plot"])
test_ds = test_ds.map(tokenize, batched=True, remove_columns=["plot"])


## Model + Trainer

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    return {
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_samples": f1_score(labels, preds, average="samples", zero_division=0),
    }

args = TrainingArguments(
    output_dir="./out",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)


## Train

In [ ]:
trainer.train()


## Evaluate on test

In [ ]:
test_metrics = trainer.evaluate(test_ds, metric_key_prefix="test")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

predictions = trainer.predict(test_ds)
probs = 1 / (1 + np.exp(-predictions.predictions))
preds = (probs >= 0.5).astype(int)
labels_arr = predictions.label_ids.astype(int)

p, r, f1, support = precision_recall_fscore_support(labels_arr, preds, average=None, zero_division=0)
per_genre = pd.DataFrame(
    {"genre": LABELS, "precision": p, "recall": r, "f1": f1, "support": support}
).sort_values("support", ascending=False)
print("\nPer-genre on test:")
print(per_genre.to_string(index=False))


## Bootstrap 95% CI on macro-F1

In [ ]:
rng = np.random.default_rng(SEED)
N = len(labels_arr)
B = 1000
boots = np.empty(B)
for b in range(B):
    idx = rng.integers(0, N, size=N)
    boots[b] = f1_score(labels_arr[idx], preds[idx], average="macro", zero_division=0)

macro_f1 = f1_score(labels_arr, preds, average="macro", zero_division=0)
lo, hi = np.percentile(boots, [2.5, 97.5])
print(f"macro-F1 = {macro_f1:.4f}  [95% CI {lo:.4f}, {hi:.4f}]")


## Push to Hub

Saves the best checkpoint plus tokenizer to `./final`, writes a model card with the eval numbers, and uploads the folder to `kolodkin/imdb-genre-distilbert`.

In [ ]:
from huggingface_hub import HfApi

per_genre_md = per_genre.to_markdown(index=False, floatfmt=".4f")
labels_md = ", ".join(f"`{l}`" for l in LABELS)

model_card = f"""---
license: mit
library_name: transformers
base_model: {MODEL_NAME}
datasets:
- kolodkin/imdb-wikipedia-enriched
pipeline_tag: text-classification
tags:
- text-classification
- multi-label
- genre-classification
- imdb
language:
- en
---

# IMDb Plot → Genre Classifier

Fine-tuned `{MODEL_NAME}` for multi-label genre classification on movie/show plot summaries from [`kolodkin/imdb-wikipedia-enriched`](https://huggingface.co/datasets/kolodkin/imdb-wikipedia-enriched).

## Labels

{labels_md}

## Test results

| metric | value |
|---|---|
| macro-F1 | {test_metrics['test_f1_macro']:.4f} |
| micro-F1 | {test_metrics['test_f1_micro']:.4f} |
| samples-F1 | {test_metrics['test_f1_samples']:.4f} |

95% bootstrap CI on macro-F1: [{lo:.4f}, {hi:.4f}] (1000 resamples).

### Per-genre

{per_genre_md}

## Usage

```python
from transformers import pipeline

clf = pipeline("text-classification", model="{HF_REPO}", top_k=None)
clf("A young wizard discovers a magical school of witchcraft.")
```

## Training

- Multi-label stratified 80 / 10 / 10 split (seed {SEED}), deduped on normalized plot hash
- {NUM_EPOCHS} epochs, lr {LR}, batch size {BATCH_SIZE}, max length {MAX_LENGTH}
- Best checkpoint selected by val macro-F1
- Source notebook: https://github.com/kolodkin/samples/blob/main/imdb-genre-distilbert/notebook.ipynb
"""

trainer.save_model("./final")
tokenizer.save_pretrained("./final")
with open("./final/README.md", "w") as f:
    f.write(model_card)

api = HfApi()
api.create_repo(HF_REPO, exist_ok=True, repo_type="model")
api.upload_folder(
    folder_path="./final",
    repo_id=HF_REPO,
    commit_message=f"Fine-tune {MODEL_NAME} on {DATASET_NAME}",
)
print(f"\nPushed to: https://huggingface.co/{HF_REPO}")


## Try the model

In [ ]:
from transformers import pipeline

clf = pipeline(
    "text-classification",
    model="./final",
    tokenizer="./final",
    top_k=None,
    device=0 if torch.cuda.is_available() else -1,
)

samples = [
    "A young wizard discovers a magical school of witchcraft and faces a dark sorcerer.",
    "Two cops in 1970s Los Angeles investigate a string of brutal murders linked to organized crime.",
    "A robot from the future is sent back in time to protect a teenager from killer machines.",
    "A struggling stand-up comedian falls in love with a journalist while touring small clubs.",
]
for s in samples:
    out = clf(s)[0]
    top3 = sorted(out, key=lambda x: -x["score"])[:3]
    pretty = ", ".join(f"{x['label']} ({x['score']:.2f})" for x in top3)
    print(f"- {s}\n  → {pretty}\n")
